# Multilingual Voice AI Assistant with LangChain RAG (for IT Support)

This notebook keeps the original multilingual voice assistant flow and adds a lightweight LangChain RAG pipeline using `RAGDataset.txt` as a local IT/customer-support knowledge base.


## Setup and dependencies


In [ ]:
!pip install openai langchain langchain-community langchain-openai faiss-cpu langdetect googletrans==4.0.0-rc1 gTTS SpeechRecognition pygame numpy


## Imports and configuration


In [ ]:
import os
import time
import speech_recognition as sr
from googletrans import Translator
from gtts import gTTS
from langdetect import detect
from openai import OpenAI

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.prompts import PromptTemplate

# Optional audio playback
try:
    import pygame
    pygame.mixer.init()
    AUDIO_READY = True
except Exception:
    AUDIO_READY = False

# Feature flag: keep simple option to switch RAG on/off
USE_RAG = True

# If key not set in environment, you can use getpass safely for this session
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI API key: ")

if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is not set. Set it in terminal or uncomment getpass lines.")

recognizer = sr.Recognizer()
translator = Translator()
client = OpenAI()


## Load knowledge base


In [ ]:
def load_knowledge_base(file_path="RAGDataset.txt"):
    """Load support knowledge base text file as LangChain documents."""
    if not os.path.exists(file_path):
        print(f"Knowledge base file not found: {file_path}")
        return []
    loader = TextLoader(file_path, encoding="utf-8")
    documents = loader.load()
    print(f"Loaded {len(documents)} document(s) from {file_path}")
    return documents


## Document chunking


In [ ]:
def split_documents(documents, chunk_size=500, chunk_overlap=80):
    """Split documents into smaller chunks for retrieval."""
    if not documents:
        return []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunk(s)")
    return chunks


## Embeddings and vector store


In [ ]:
def build_vector_store(chunks):
    """Create FAISS vector store from chunks using OpenAI embeddings."""
    if not chunks:
        return None
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vector_store = FAISS.from_documents(chunks, embeddings)
    print("Vector store built successfully")
    return vector_store


def retrieve_context(query, retriever, k=3):
    """Retrieve top-k relevant chunks for a query."""
    if retriever is None:
        return []
    docs = retriever.invoke(query)
    return docs[:k]


## Input capture


In [ ]:
def take_input():
    """Try microphone first, then fallback to typed input."""
    try:
        with sr.Microphone() as source:
            print("Listening...")
            recognizer.adjust_for_ambient_noise(source)
            audio = recognizer.listen(source, timeout=8, phrase_time_limit=10)
        text = recognizer.recognize_google(audio)
        print("You said:", text)
        return text
    except Exception:
        return input("Type your question: ").strip()


## Language detection and translation


In [ ]:
def translate_to_english(text):
    source_lang = detect(text)
    english_text = translator.translate(text, dest="en").text
    return source_lang, english_text


def translate_from_english(text, target_lang):
    return translator.translate(text, dest=target_lang).text


## Grounded response generation


In [ ]:
grounded_prompt = PromptTemplate.from_template(
    """You are a helpful IT customer-support assistant.
Use the retrieved context as the primary source of truth.
If context is not enough, clearly say you do not have enough information from the knowledge base.
Do not invent policies, configurations, or troubleshooting steps.
Provide concise and practical help.


Retrieved context:
{context}

User question (English):
{question}

Grounded answer (English):"""
)


def generate_grounded_response(query, context_text, use_rag=True):
    if use_rag and context_text.strip():
        prompt_text = grounded_prompt.format(context=context_text, question=query)
    else:
        prompt_text = f"""You are a helpful IT support assistant.
Answer clearly in simple language.

User question (English): {query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an IT support assistant."},
            {"role": "user", "content": prompt_text}
        ],
        temperature=0.3,
        max_tokens=220
    )
    return response.choices[0].message.content or ""


## Translation back to original language


In [ ]:
def translate_answer_back(answer_english, user_lang):
    return translate_from_english(answer_english, user_lang)


## Text-to-speech output


In [ ]:
def speak_response(text, lang):
    """Convert text to speech and play it when audio is available."""
    if not AUDIO_READY:
        print("(Audio not available in this environment)")
        return

    file_name = "response_voice.mp3"
    try:
        gTTS(text=text, lang=lang, slow=False).save(file_name)
        pygame.mixer.Sound(file_name).play()
        while pygame.mixer.get_busy():
            time.sleep(0.3)
    except Exception as e:
        print("Audio playback failed:", e)
    finally:
        if os.path.exists(file_name):
            os.remove(file_name)


## Main execution flow


In [ ]:
# Build RAG resources once
kb_docs = load_knowledge_base("RAGDataset.txt")
kb_chunks = split_documents(kb_docs)
vector_store = build_vector_store(kb_chunks)
retriever = vector_store.as_retriever(search_kwargs={"k": 3}) if vector_store else None

# Input
user_query = take_input()
if not user_query:
    print("No input provided.")
else:
    # Language + translation
    user_lang, english_query = translate_to_english(user_query)
    print("Detected language:", user_lang)
    print("English query:", english_query)

    # RAG retrieval
    retrieved_docs = retrieve_context(english_query, retriever, k=3) if USE_RAG else []
    context_preview_list = []
    for i, d in enumerate(retrieved_docs, 1):
        snippet = d.page_content[:220].replace("\n", " ")
        context_preview_list.append(f"[{i}] {snippet}")

    context_text = "\n\n".join([d.page_content for d in retrieved_docs]) if retrieved_docs else ""

    print("\nRetrieved chunk previews:")
    if context_preview_list:
        for p in context_preview_list:
            print(p)
    else:
        print("(No retrieved context)")

    # Grounded generation
    answer_en = generate_grounded_response(english_query, context_text, use_rag=USE_RAG)
    print("\nGrounded English answer:", answer_en)

    # Translate back + speak
    final_answer = translate_answer_back(answer_en, user_lang)
    print("\nFinal translated response:", final_answer)
    speak_response(final_answer, user_lang)


## Project explanation and limitations

### What was originaly done
- Captured voice input (with text fallback), detected language, translated to English, asked an OpenAI model, translated the answer back, and optionally played audio.

### What changed after adding RAG
- A real retrieval step now runs between translation and answer generation.
- The notebook loads `RAGDataset.txt`, splits it into chunks, embeds them, stores them in FAISS, and retrieves top matches per query.

### How LangChain fits the pipeline
- LangChain is used for loader, splitting, embeddings, and vector retrieval.
- OpenAI chat completion is used for final grounded answer generation.

### Multilingual IT support use case
- Users can ask in Hindi/Spanish/other languages.
- The assistant retrieves relevant internal support knowledge and answers in the user's language.

### Limitations
- Demo quality depends on the content and size of `RAGDataset.txt`.
- Translation and speech recognition quality vary by language/accent.
